# Notebook 07: Fixed Anatomical Graph Neural Network (Fixed GNN)

**Objective**: Train and evaluate the Fixed Anatomical Graph Neural Network (`FixedAnatomicalGNN`).

### Architecture Overview:
1. **Regional Temporal Encoders**: Separate 1D Conv/BatchNorm/Pooling blocks for each of the 5 body regions, transforming variable-channel sensor streams into uniform $D$-dimensional node representations ($D=64$).
2. **Fixed Graph Convolution**: Standard Kipf-Welling GCN message passing over the static skeletal adjacency matrix $\tilde{A}_{\text{fixed}}$ connecting limbs to the trunk.
3. **Global Graph Readout**: Concatenated mean and max pooling across all 5 anatomical nodes followed by an MLP classifier.

> **Compute Rule**: Full training requires CUDA GPU (Colab). Set `EXPERIMENT_MODE = 'DEBUG'` for 1-batch CPU verification.

## 1. Configure Experiment Mode

In [ ]:
# Choose: 'DEBUG' (1-epoch CPU/GPU check), 'FAST' (5 epochs GPU), or 'FULL' (50 epochs GPU)
EXPERIMENT_MODE = "DEBUG"

import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt

current = Path.cwd()
REPO_ROOT = current.parent if current.name == "notebooks" else current
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.api as api
env_info = api.setup_project()

## 2. Load Processed Sliding Windows & DataLoaders

In [ ]:
subset = api.load_preprocessed_subset("opportunity_locomotion_subset.npz")
X_train, y_train = subset["X_train"], subset["y_train"]
X_val, y_val = subset["X_val"], subset["y_val"]
X_test, y_test = subset["X_test"], subset["y_test"]

if EXPERIMENT_MODE == "DEBUG":
    X_train, y_train = X_train[:64], y_train[:64]
    X_val, y_val = X_val[:32], y_val[:32]
    X_test, y_test = X_test[:32], y_test[:32]

train_loader, val_loader, test_loader = api.create_dataloaders(
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test,
    batch_size=64 if EXPERIMENT_MODE != "DEBUG" else 16,
)
print(f"DataLoaders: {len(train_loader)} train batches, {len(val_loader)} val batches.")

## 3. Instantiate Fixed Anatomical GNN

In [ ]:
fixed_gnn = api.build_fixed_gnn(
    num_classes=5,
    sequence_length=30,
    node_dim=64,
    gcn_hidden_dim=64,
    dropout=0.2
)

total_params = sum(p.numel() for p in fixed_gnn.parameters() if p.requires_grad)
print(f"FixedAnatomicalGNN Initialized! Trainable parameters: {total_params:,}")

## 4. Train Model (Automatic GPU/CUDA Guard)

In [ ]:
best_fixed_gnn, history = api.train_model(
    model=fixed_gnn,
    train_loader=train_loader,
    val_loader=val_loader,
    mode=EXPERIMENT_MODE,
    lr=1e-3,
    experiment_name="fixed_anatomical_gnn"
)
print("Training Complete!")

## 5. Evaluate on Held-Out Test Set

In [ ]:
test_metrics = api.evaluate_model(best_fixed_gnn, test_loader)

print("\n" + "=" * 55)
print(" FIXED ANATOMICAL GNN TEST EVALUATION")
print("=" * 55)
print(f"Accuracy    : {test_metrics['accuracy'] * 100:.2f}%")
print(f"Macro F1    : {test_metrics['macro_f1'] * 100:.2f}%")
print(f"Weighted F1 : {test_metrics['weighted_f1'] * 100:.2f}%")
print("=" * 55)